In [2]:
# !pip install transformers accelerate gradio pandas openpyxl bitsandbytes huggingface_hub

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig
import torch
import gradio as gr
import pandas as pd
from huggingface_hub import login

# 🔐 Hugging Face Login
login("tokken")  # Replace with your HF token

# ✅ Quantization Config (4-bit)
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

# ✅ Load model + tokenizer with quantization
model_id = "mistralai/Mistral-7B-Instruct-v0.1"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quant_config,
    device_map="auto",  # automatically uses GPU if available
)

# Create text generation pipeline
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    torch_dtype=torch.float16,
    device_map="auto",
    max_new_tokens=1024,
)

# Dataset generator logic
def generate_dataset(prompt):
    instruction = f"[INST] {prompt}. Output only the data in CSV format. [/INST]"
    result = generator(instruction)[0]["generated_text"]

    # Extract and format as CSV
    lines = result.strip().split("\n")
    rows = [line.strip().split(",") for line in lines if "," in line]

    if not rows or len(rows[0]) < 2:
        return "❌ Model didn't return a valid table. Try a simpler prompt."

    df = pd.DataFrame(rows)
    file_path = "mistral_quantized_dataset.xlsx"
    df.to_excel(file_path, index=False, header=False)
    return file_path

# Launch GUI
iface = gr.Interface(
    fn=generate_dataset,
    inputs=gr.Textbox(
        lines=3,
        label="Enter your prompt",
        placeholder="e.g. Generate 50 employees: Name, Age, Dept, ID, Salary"
    ),
    outputs=gr.File(label="Download Excel File (.xlsx)"),
    title="📊 Mistral 7B (4-bit Quantized) Dataset Generator",
    description="Runs Mistral 7B on GPU with 4-bit quantization. Generate datasets from natural language prompts."
)

iface.launch(share=True)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda:0


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://36833b11f39d4ad7e4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [2]:
!pip install bitsandbytes


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 114.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 62.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 79.8 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstallin